# Chest X-ray Dataset Formatting and Classification

To format the dataset we will:
1. **Resize** all images to `256x256`
2. **Classify** images into an organized directory structure:
   - `dataset/train/normal` — normal images (train)
   - `dataset/train/bacteria` — bacterial pneumonia (train)
   - `dataset/train/virus` — viral pneumonia (train)
   - `dataset/test/normal` — normal images (test)
   - `dataset/test/bacteria` — bacterial pneumonia (test)
   - `dataset/test/virus` — viral pneumonia (test)
3. **Count** images before and after processing to ensure nothing is lost

## Imports

Required libraries:
- `numpy` for array manipulation
- `PIL.Image` for image resizing
- `os` for file system navigation
- `sys` for error handling

In [ ]:
import numpy as np
import os
from PIL import Image
import sys

## `resize_image(image_path, output_path, dim1, dim2)`

Resizes an image to dimensions `dim1 x dim2` and saves it to `output_path`.
It also convert images to 1 canal (only Grey) with the line `img = img.convert('L')`.
after that the dataset is fully uniformised.

**Parameters:**
- `image_path`: path to the source image
- `output_path`: destination path (directories are created automatically)
- `dim1`, `dim2`: target dimensions (width, height)

**Returns:** the path of the saved image

In [ ]:
def resize_image(image_path, output_path, dim1, dim2):
    print("Resizing image:", image_path)
    print("Output path:", output_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    img = Image.open(image_path)
    img = img.convert('L')
    img = img.resize((dim1, dim2))
    img.save(output_path)
    return output_path

## `classify_image(filename, source_subdir)`

Determines the destination subdirectory for an image based on its filename and source directory.

**Classification logic:**
- If the image comes from `NORMAL` → `normal` folder
- If the image comes from `PNEUMONIA` and contains **"bacteria"** in its name → `bacteria` folder
- If the image comes from `PNEUMONIA` and contains **"virus"** in its name → `virus` folder
- Otherwise → `None` (image skipped)

**Returns:** the destination subdirectory name (`"normal"`, `"bacteria"`, `"virus"`) or `None`

In [ ]:
def classify_image(filename, source_subdir):
    """Determines the destination folder based on the filename and its source directory."""
    name_lower = filename.lower()
    if source_subdir == "NORMAL":
        return "normal"
    elif source_subdir == "PNEUMONIA":
        if "bacteria" in name_lower:
            return "bacteria"
        elif "virus" in name_lower:
            return "virus"
    return None

## `count_images(filepath)`

Recursively counts the number of images (`.png`, `.jpg`, `.jpeg`) in a given directory.

**Parameter:** `filepath` — path of the directory to scan

**Returns:** the total number of images found

In [ ]:
def count_images(filepath):
    """Compte le nombre d'images dans un dossier récursivement."""
    count = 0
    for root, dirs, filenames in os.walk(filepath):
        for filename in filenames:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                count += 1
    return count

## `format_images(filepath, output_base)`

Main processing function. Iterates through the `train/` and `test/` directories of the source dataset and for each image:
1. Determines its label via `classify_image()`
2. Resizes it to **256x256** via `resize_image()`
3. Saves it to the appropriate destination folder

**Parameters:**
- `filepath`: path to the source dataset (containing `train/` and `test/`)
- `output_base`: root output path (e.g., `./dataset`)

**Returns:** a `np.array` containing the paths of all processed images

In [ ]:
def format_images(filepath, output_base):
    images = []

    if not os.path.exists(filepath):
        sys.stderr.write(f"Error: Directory '{filepath}' does not exist.\n")
        sys.exit(84)

    for split in ["train", "test"]:
        split_path = os.path.join(filepath, split)
        if not os.path.exists(split_path):
            print(f"Directory '{split}' not found")
            continue

        for category in ["NORMAL", "PNEUMONIA"]:
            category_path = os.path.join(split_path, category)
            if not os.path.exists(category_path):
                print(f"Directory '{split}/{category}' not found")
                continue

            for filename in os.listdir(category_path):
                if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                    continue
                dest_subdir = classify_image(filename, category)

                if dest_subdir is None:
                    print(f"Image skipped: {filename}")
                    continue

                source_path = os.path.join(category_path, filename)
                output_path = os.path.join(output_base, split, dest_subdir, filename)

                try:
                    result = resize_image(source_path, output_path, 256, 256)
                    images.append(result)
                except Exception as e:
                    sys.stderr.write(f"Error processing image {filename}: {e}\n")
                    sys.exit(84)

    return np.array(images) if images else np.array([])

## `main()`

Entry point of the script, runs in this order:
1. **Pre-processing count** — displays the number of images per source directory (`train/NORMAL`, `train/PNEUMONIA`, `test/NORMAL`, `test/PNEUMONIA` and total)
2. **Processing** — calls `format_images` to resize and classify all images
3. **Post-processing count** — displays the number of images per destination directory (`train/normal`, `train/bacteria`, etc.) to verify all images were processed

In [ ]:
def main():
    dataset_folder = os.path.abspath('./chest_xray')
    output_folder = os.path.abspath('./dataset')

    print("IMAGE COUNT BEFORE PROCESSING")
    total_before = 0
    for split in ["train", "test"]:
        split_path = os.path.join(dataset_folder, split)
        if os.path.exists(split_path):
            for category in ["NORMAL", "PNEUMONIA"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_before += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL before: {total_before} images")

    formatted_images = format_images(dataset_folder, output_folder)
    print(f"\nNumber of formatted images: {len(formatted_images)}")

    print("IMAGE COUNT AFTER PROCESSING")
    total_after = 0
    for split in ["train", "test"]:
        split_path = os.path.join(output_folder, split)
        if os.path.exists(split_path):
            for category in ["normal", "bacteria", "virus"]:
                cat_path = os.path.join(split_path, category)
                if os.path.exists(cat_path):
                    n = count_images(cat_path)
                    total_after += n
                    print(f"  {split}/{category}: {n} images")
    print(f"  TOTAL after: {total_after} images")

if __name__ == "__main__":
    main()